# Lab W3: Ablation Loss × Freeze

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `03_W3_Loss_Optimizer_Evaluasi.md` dan `04_W4_Reproducibility_Experiment_Matrix.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W3 sudah dibaca, terutama §2.1 (loss matching, focal loss formula `(1-p_t)^γ`).
- Bab W4 sudah dibaca, terutama §2.0 (matriks eksperimen), §2.5 (hipotesis falsifiable), §2.4 (seed variance + Aturan 2σ).
- Lab W2 (`lab_w2_cnn_baseline.ipynb`) sudah selesai - SimpleCNN baseline jalan.

**Yang akan Anda hasilkan di akhir lab:**
- Pre-registration `docs/preregs/lab2_<tanggal>.md` di-commit **sebelum** training pertama.
- 4 kondisi × 3 seed = 12 run dengan 15 epoch tiap run.
- Sanity check `FocalLoss(gamma=0) ≡ CrossEntropyLoss` (selisih < 0.1%).
- Tabel mean ± std + bar chart dengan error bar.
- Analisis main effect (loss, freeze) dan interaksi.
- Refleksi yang membandingkan hasil aktual vs prediksi pre-reg.

**Resource:**
- **Hardware:** GPU sangat dianjurkan; CPU 12 run × 15 epoch ~ 6-8 jam.
- **Estimasi waktu kerja:** 4-6 jam termasuk pre-reg, eksekusi, analisis, refleksi.

**Pendamping:** Bab W3 dan W4.

## Skenario
Instruksi PI: *"Tolong uji focal loss dan freeze blok awal pada backbone. Bandingkan dengan baseline yang adil, lalu kirim ringkasan hasil hari Kamis."*
Terjemahkan ke grid 2×2: loss ∈ {CE, Focal} × freeze ∈ {None, block1}, 3 seed tiap sel.

## Tujuan
1. Tulis pre-registration sebelum menjalankan satu pun eksperimen.
2. Jalankan 4 kondisi × 3 seed = 12 run (15 epoch tiap run untuk hemat waktu).
3. Rangkum hasil di tabel dengan mean ± std dan per-class F1.
4. Interpretasi main effect dan interaksi.

## Checklist
- [ ] Pre-registration di `docs/preregs/lab2_<tanggal>.md` sebelum kode eksperimen dijalankan.
- [ ] 12 run lengkap, log dan checkpoint tersimpan.
- [ ] Tabel hasil dengan mean ± std.
- [ ] Bar chart dengan error bar.
- [ ] Paragraf interpretasi main effect + interaksi.
- [ ] Verifikasi: `FocalLoss(gamma=0)` reproduksi CE baseline (< 0.1% selisih).

## 0. Pre-registration

**BUAT DULU sebelum jalankan eksperimen.** Salin `docs/prereg_template.md`, isi, dan commit.

```bash
cp docs/prereg_template.md docs/preregs/lab2_$(date +%Y%m%d).md
# edit file, lalu:
git add docs/preregs/
git commit -m "prereg: Lab 2 focal loss vs CE ablation"
```

Tempel path dan commit hash pre-reg di sini setelah selesai:

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo && pip install -e .
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

> Pre-reg path: `docs/preregs/lab2_YYYYMMDD.md`  
> Pre-reg commit: `[hash]`  
> Hipotesis utama: *[salin dari pre-reg]*

## 1. Verifikasi FocalLoss(gamma=0) ≡ CrossEntropyLoss

Ini sanity check yang wajib sebelum eksperimen penuh.

In [ ]:
import torch
from src.losses import FocalLoss

torch.manual_seed(0)
logits = torch.randn(16, 10)
labels = torch.randint(0, 10, (16,))

ce_loss = torch.nn.CrossEntropyLoss()(logits, labels).item()
focal_loss = FocalLoss(gamma=0.0)(logits, labels).item()

diff = abs(ce_loss - focal_loss)
print(f'CrossEntropy:  {ce_loss:.6f}')
print(f'Focal(γ=0):    {focal_loss:.6f}')
print(f'Selisih:       {diff:.6f}')
assert diff < 1e-4, f'FocalLoss(gamma=0) != CE! Selisih: {diff}'
print('✓ Sanity check lulus: FocalLoss(gamma=0) ≡ CrossEntropyLoss')

import yaml
from pathlib import Path
import itertools

# Grid
losses = ['cross_entropy', 'focal']
freezes = [None, 'block1']
seeds = [42, 123, 2024]

# Baca baseline config sebagai template
with open('../configs/baseline.yaml') as f:
    base_cfg = yaml.safe_load(f)

config_dir = Path('../configs/lab2')
config_dir.mkdir(exist_ok=True)

configs = []
for loss_name, freeze_until in itertools.product(losses, freezes):
    cfg = yaml.safe_load(yaml.dump(base_cfg))  # deep copy
    freeze_label = freeze_until or 'none'
    exp_name = f'lab2_{loss_name}_freeze{freeze_label}'

    cfg['experiment_name'] = exp_name
    cfg['train']['epochs'] = 15  # lebih pendek untuk lab
    cfg['loss'] = {'name': loss_name}
    if loss_name == 'focal':
        cfg['loss']['gamma'] = 2.0
    cfg['model']['freeze_until'] = freeze_until

    config_path = config_dir / f'{exp_name}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    for seed in seeds:
        configs.append({'loss': loss_name, 'freeze': freeze_label, 'seed': seed,
                        'config_path': str(config_path), 'exp_name': exp_name})

print(f'Total run: {len(configs)}')
for c in configs:
    print(f"  loss={c['loss']:15s} freeze={c['freeze']:7s} seed={c['seed']}")

In [ ]:
import yaml
from pathlib import Path
import itertools

# Grid
losses = ['cross_entropy', 'focal']
freezes = [None, 'block1']
seeds = [42, 123, 2024]

# Baca baseline config sebagai template
with open('configs/baseline.yaml') as f:
    base_cfg = yaml.safe_load(f)

config_dir = Path('configs/lab2')
config_dir.mkdir(exist_ok=True)

configs = []
for loss_name, freeze_until in itertools.product(losses, freezes):
    cfg = yaml.safe_load(yaml.dump(base_cfg))  # deep copy
    freeze_label = freeze_until or 'none'
    exp_name = f'lab2_{loss_name}_freeze{freeze_label}'

    cfg['experiment_name'] = exp_name
    cfg['train']['epochs'] = 15  # lebih pendek untuk lab
    cfg['loss'] = {'name': loss_name}
    if loss_name == 'focal':
        cfg['loss']['gamma'] = 2.0
    cfg['model']['freeze_until'] = freeze_until

    config_path = config_dir / f'{exp_name}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    for seed in seeds:
        configs.append({'loss': loss_name, 'freeze': freeze_label, 'seed': seed,
                        'config_path': str(config_path), 'exp_name': exp_name})

print(f'Total run: {len(configs)}')
for c in configs:
    print(f"  loss={c['loss']:15s} freeze={c['freeze']:7s} seed={c['seed']}")

import subprocess, sys, os

RUN_NOW = False  # Ganti ke True untuk jalankan dari notebook (lambat)

if RUN_NOW:
    seen_configs = set()
    for c in configs:
        if c['config_path'] not in seen_configs:
            seen_configs.add(c['config_path'])
        for seed in seeds:
            cmd = [sys.executable, '-m', 'src.train',
                   '--config', c['config_path'], '--seed', str(seed)]
            print(f'Running: {" ".join(cmd)}')
            result = subprocess.run(cmd, capture_output=True, text=True,
                                    cwd=os.path.abspath('..'))
            if result.returncode != 0:
                print('ERROR:', result.stderr[-500:])
    print('Semua run selesai.')
else:
    print('RUN_NOW=False. Jalankan dari terminal di folder template_repo/, lalu lanjutkan ke sel berikutnya.')

In [ ]:
import subprocess, sys

RUN_NOW = False  # Ganti ke True untuk jalankan dari notebook (lambat)

if RUN_NOW:
    seen_configs = set()
    for c in configs:
        if c['config_path'] not in seen_configs:
            seen_configs.add(c['config_path'])
        for seed in seeds:
            cmd = [sys.executable, '-m', 'src.train',
                   '--config', c['config_path'], '--seed', str(seed)]
            print(f'Running: {" ".join(cmd)}')
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0:
                print('ERROR:', result.stderr[-500:])
    print('Semua run selesai.')
else:
    print('RUN_NOW=False. Jalankan dari terminal, lalu lanjutkan ke sel berikutnya.')

## 4. Rangkum hasil

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../experiments/results.csv')

# Filter baris Lab 2
df_lab2 = df[df['experiment_name'].str.startswith('lab2_')].copy()

# Parse nama eksperimen ke kolom terpisah
def parse_exp(name):
    # format: lab2_{loss}_freeze{freeze}
    parts = name.replace('lab2_', '').rsplit('_freeze', 1)
    return parts[0], parts[1] if len(parts) > 1 else 'none'

df_lab2[['loss', 'freeze']] = df_lab2['experiment_name'].apply(
    lambda x: pd.Series(parse_exp(x))
)
df_lab2['best_val_acc'] = df_lab2['best_val_acc'].astype(float)

# Agregasi
summary = df_lab2.groupby(['loss', 'freeze'])['best_val_acc'].agg(['mean', 'std', 'count'])
summary.columns = ['mean_val_acc', 'std_val_acc', 'n_seeds']
summary = summary.reset_index()
summary['mean_pct'] = summary['mean_val_acc'] * 100
summary['std_pct'] = summary['std_val_acc'] * 100

print('Tabel hasil Lab 2:')
print(summary.to_string(index=False))

## 5. Bar chart dengan error bar

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

conditions = [f"{r['loss']}\n+freeze={r['freeze']}" for _, r in summary.iterrows()]
means = summary['mean_pct'].values
stds = summary['std_pct'].values

colors = ['#4C8BB5', '#E07B54', '#5BAD6F', '#C25B99']
bars = ax.bar(conditions, means, yerr=stds, capsize=6,
              color=colors[:len(conditions)], alpha=0.85, edgecolor='white', linewidth=1.5)

# Baseline reference line (CE, no freeze)
baseline_row = summary[(summary['loss'] == 'cross_entropy') & (summary['freeze'] == 'none')]
if not baseline_row.empty:
    baseline_mean = baseline_row['mean_pct'].values[0]
    ax.axhline(baseline_mean, color='gray', linestyle='--', alpha=0.6, label=f'Baseline: {baseline_mean:.1f}%')

# Annotate values
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.3,
            f'{mean:.1f}±{std:.1f}', ha='center', va='bottom', fontsize=9)

ymin = max(0, min(means) - max(stds) - 3)
ymax = max(means) + max(stds) + 3
ax.set_ylim(ymin, ymax)
ax.set_ylabel('Val Accuracy (%) — mean ± std dari 3 seed')
ax.set_title('Lab 2: Ablation Loss × Freeze')
ax.legend()
ax.grid(axis='y', alpha=0.3)

output_dir = Path('../experiments')
(output_dir / 'plots').mkdir(exist_ok=True)
plt.tight_layout()
plt.savefig(output_dir / 'plots/lab2_ablation.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Analisis main effect dan interaksi

In [ ]:
# Main effect loss: rata-rata seluruh freeze conditions
effect_loss = summary.groupby('loss')['mean_pct'].mean()
print('Main effect loss:')
print(effect_loss.to_string())
print(f'  → Δ(focal - CE): {effect_loss.get("focal", 0) - effect_loss.get("cross_entropy", 0):+.2f}%')

print()

# Main effect freeze
effect_freeze = summary.groupby('freeze')['mean_pct'].mean()
print('Main effect freeze:')
print(effect_freeze.to_string())
print(f'  → Δ(freeze=block1 - none): {effect_freeze.get("block1", 0) - effect_freeze.get("none", 0):+.2f}%')

print()

# Interaksi: apakah Focal+Freeze berbeda dari jumlah dua main effects?
pivot = summary.pivot(index='freeze', columns='loss', values='mean_pct')
print('Pivot table (mean val_acc %):')
print(pivot.round(2))

if 'focal' in pivot.columns and 'cross_entropy' in pivot.columns:
    diff_no_freeze = pivot.loc['none', 'focal'] - pivot.loc['none', 'cross_entropy']
    diff_freeze = pivot.loc['block1', 'focal'] - pivot.loc['block1', 'cross_entropy']
    interaction = diff_freeze - diff_no_freeze
    print(f'\nEfek focal (tanpa freeze): {diff_no_freeze:+.2f}%')
    print(f'Efek focal (dengan freeze): {diff_freeze:+.2f}%')
    print(f'Interaksi: {interaction:+.2f}%')
    if abs(interaction) > 1.0:
        print('⚠ Ada interaksi yang cukup besar — dua variabel tidak independen.')
    else:
        print('✓ Efek hampir aditif — interaksi kecil.')

## 7. Refleksi

1. Apa *main effect* dari mengganti loss saja (focal vs CE), rata-rata atas semua kondisi freeze? Apakah perbedaannya lebih besar dari 2× std antar-seed?

2. Apakah ada *interaksi* antara loss dan freeze — artinya, apakah efek focal berbeda tergantung ada/tidaknya freeze? Apa implikasinya?

3. Jika hasilmu *inconclusive* (perbedaan kecil dibanding noise seed-to-seed), apa satu perubahan pada desain eksperimen yang akan memberikan jawaban lebih tegas?

4. Bandingkan hasil aktual dengan yang kamu tulis di pre-registration. Di mana prediksimu tepat, dan di mana meleset?

### Jawaban Refleksi

**1. Main effect loss:**
> *[tulis di sini]*

**2. Interaksi loss × freeze:**
> *[tulis di sini]*

**3. Perbaikan desain eksperimen:**
> *[tulis di sini]*

**4. Perbandingan dengan pre-registration:**
> *[tulis di sini]*